In [ ]:
# Packages to Install for Scraping
!pip -q install requests beautifulsoup4 
import requests, json
from bs4 import BeautifulSoup
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import hashlib
import os

# Get the base URL for the Notices (And URL for archives)
notice_landing = "https://www.boston.gov/public-notices"
archive_landing = "https://www.boston.gov/archived-public-notices"

# Ensure that the path for the PDFs exists
folder_name = "public-notice-pdfs"
notice_folder = os.path.join(folder_name, str(notice_id))
os.makedirs(notice_folder, exist_ok=True)

In [ ]:
# Function to write log
def log_notice(log_path:str, record):
    with open(path,"a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False)+"\n")


#Function to filter log to check file ids
def check_logs(log_path:str, notice_id:str):
    ''' Returns the last record for the given notice id if it exists, otherwise None'''
    with open(log_path, encoding="utf-8") as f:
        lines = f.readlines()

    for line in reversed(lines):
        record = json.loads(line)
        if str(record.get("notice_id")) == notice_id:
            return record
    return None
    
# Function to get text safely when scraping
def safe_get_text(container, tag, **kwargs):
    ''' Get text safely if there's an empty field'''
    found = container.find(tag, **kwargs)
    return found.get_text(strip=False) if found else ""

# Functions to extract data from a Notice 
def extract_notice(notice_id:str, log_path:str):
    # Get the link for the notice:
    notice_url = notice_landing+"/"+notice_id

    #Get the url contents
    response = requests.get(notice_Url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Start extracting 
    title=soup.title.string
    
    # Finding the Posted Date
    posted_label=soup.find("div",class_="dl-t", string=lambda t:t and "Posted" in t)
    posted_raw=posted_label.find_next_sibling("div",class_="dl-d").get_text(strip=True)
    posted_at = datetime.strptime(posted_raw, "%m/%d/%Y - %I:%M%p").replace(tzinfo=ZoneInfo("America/New_York")).isoformat()

    # Discussion Topics Text
    discussion_label = soup.find("h2",class_="header-border-bottom", string=lambda t:t and "Discussion Topics" in t)
    discussion_text = discussion_label.find_next_sibling("div",class_="body").get_text(strip=False)


    # Event details
    event_date_container = soup.find("div", class_="date-title")
    event_datetime = event_date_container.find("time")["datetime"]
    address_container = soup.find("div",class_="detail-item__body--secondary sb-d")
    address_line_1 = safe_get_text(address_container, "span", class_="address-line1")
    address_line_2 = safe_get_text(address_container, "span", class_="address-line2")

    #Look for public comment
    public_testimony = False
    testimony = soup.find("div",class_="n-li-a", string=lambda t:t and "The public can offer testimony" in t)
    if testimony:
        public_testimony = True

    # Look for cancellation
    cancelled = False
    cancellation = soup.find("span",class_="t--err t--s60pct", string=lambda t:t and "Canceled" in t)
    if cancellation:
        cancelled=True
    
    # PDFS
    files = []
    resources_label = soup.find("div", class_="sb-t", string=lambda t: t and "Resources" in t)
    resources_container = resources_label.find_parent("div", class_="detail-item__content")
    pdf_links = resources_container.select("div.link-wrapper.download-link a")

    files = [{"file_label": a.get_text(strip=True), "file_url": a["href"]} for a in pdf_links]


    # Check if any files have been added
    ## Get the last record
    previous = check_logs(log_path, notice_id)

    if previous:
        # Have to check with old record
        old_files = [{
            "file_label": f["file_label"],
            "file_url": f["file_url"],
            "download_success": f.get("download_success", False),
        } for f in previous.get("files", [])
        ]
        old_urls = {f["file_url"] for f in old_files}
        new_urls = {f["file_url"] for f in files}

        # Files that are brand new to the page
        added_files = [f for f in new_files if f["file_url"] not in old_urls]

        # Files that disappeared from the page
        removed_files = [f for f in old_files if f["file_url"] not in new_urls]
        # Files that were on the page before AND still are, but never downloaded successfully
        failed_previously = {f["file_url"] for f in old_files if not f["download_success"]}

        needs_download = [
            f for f in new_files
            if f["file_url"] not in old_urls           # brand new
            or f["file_url"] in failed_previously        # or previously failed
            ]
        # Download PDFs
    else:
        # Brand new, just write
        # Download PDFS


    

In [ ]:


# Get the notice landing
landing_response = requests.get(notice_landing)
landing_soup = BeatifulSoup(response.text, 'html.parser')


# Loop through pages will eventually go here

# Loop through Pages (Notices) on a page

# 